# Maxwell Solver in 2D Using FDTD Scheme

---
## **Introduction to FDTD for Maxwell's Equations**

The **Finite-Difference Time-Domain (FDTD)** method is a **numerical technique** for solving **Maxwell's equations** in the time domain. It is widely used in **electromagnetics** to model the propagation of electromagnetic waves.

### **Maxwell's Equations in 2D**

For a 2D transverse magnetic (TM) mode, Maxwell's equations reduce to:

$$
\frac{\partial H_z}{\partial t} = \frac{\partial E_x}{\partial y} - \frac{\partial E_y}{\partial x}
$$
$$
\frac{\partial E_x}{\partial t} = \frac{\partial H_z}{\partial y}
$$
$$
\frac{\partial E_y}{\partial t} = -\frac{\partial H_z}{\partial x}
$$

![FDTD Scheme](images/fdtd.png)

**Image:** Visualization of the FDTD grid for Maxwell's equations.

---
## **FDTD Update Equations**

The FDTD method uses **staggered grids** in space and time to update the electric and magnetic fields. The update equations for the **Yee algorithm** are:

### **Faraday's Law (Magnetic Field Update)**
$$
H_z \big|^{n+1/2}_{i+1/2,j+1/2} = H_z \big|^{n-1/2}_{i+1/2,j+1/2} +
\frac{dt}{dy} \left( E_x \big|^{n}_{i+1/2,j+1} - E_x \big|^{n}_{i+1/2,j} \right)
- \frac{dt}{dx} \left( E_y \big|^{n}_{i+1,j+1/2} - E_y \big|^{n}_{i,j+1/2} \right)
$$

### **Ampère's Law (Electric Field Update)**
$$
E_x \big|^{n+1}_{i+1/2,j} = E_x \big|^{n}_{i+1/2,j} + \frac{dt}{dy} \left( H_z \big|^{n+1/2}_{i+1/2,j+1/2} - H_z \big|^{n+1/2}_{i+1/2,j-1/2} \right)
$$

$$
E_y \big|^{n+1}_{i,j+1/2} = E_y \big|^{n}_{i,j+1/2} - \frac{dt}{dx} \left( H_z \big|^{n+1/2}_{i+1/2,j+1/2} - H_z \big|^{n+1/2}_{i-1/2,j+1/2} \right)
$$

**Reference:** [Finite-Difference Time-Domain Method (Wikipedia)](https://en.wikipedia.org/wiki/Finite-difference_time-domain_method)

---
## **Implementation in Python**

To implement the FDTD scheme, we first set up the **mesh parameters** and initialize the fields.

### **Mesh Parameters and Initialization**

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import axes3d
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams['figure.figsize'] = (10, 6)

# Mesh parameters
nx, ny = 101, 101  # Number of grid points in x and y
vx, dx = np.linspace(0, 1, nx, endpoint=True, retstep=True)  # x-grid
vy, dy = np.linspace(0, 1, ny, endpoint=True, retstep=True)  # y-grid

# Initialize electric and magnetic fields
ex = np.zeros((nx-1, ny), dtype=np.double)  # E_x field
ey = np.zeros((nx, ny-1), dtype=np.double)  # E_y field
hz = np.zeros((nx-1, ny-1), dtype=np.double)  # H_z field

---
## **Fortran Subroutines for FDTD Updates**

To optimize performance, we can implement the FDTD update equations in **Fortran** and call them from Python.

---
### **Faraday's Law in Fortran**
```fortran
%%fortran
subroutine faraday_fortran(ex, ey, hz, dx, dy, dt, nx, ny)
    implicit none
    integer, intent(in) :: nx, ny
    real(8), intent(in) :: dx, dy, dt
    real(8), dimension(nx-1, ny), intent(in) :: ex
    real(8), dimension(nx, ny-1), intent(in) :: ey
    real(8), dimension(nx-1, ny-1), intent(inout) :: hz
    integer :: i, j
    real(8) :: dex_dy, dey_dx

    do j = 1, ny-1
        do i = 1, nx-1
            dex_dy = (ex(i, j+1) - ex(i, j)) / dy
            dey_dx = (ey(i+1, j) - ey(i, j)) / dx
            hz(i, j) = hz(i, j) + dt * (dex_dy - dey_dx)
        end do
    end do
end subroutine faraday_fortran
```

---
### **Ampère's Law in Fortran**
```fortran
%%fortran
subroutine amperemaxwell_fortran(ex, ey, hz, dx, dy, dt, nx, ny)
    implicit none
    integer, intent(in) :: nx, ny
    real(8), intent(in) :: dx, dy, dt
    real(8), dimension(nx-1, ny-1), intent(inout) :: hz
    real(8), dimension(nx-1, ny), intent(inout) :: ex
    real(8), dimension(nx, ny-1), intent(inout) :: ey
    integer :: i, j
    real(8) :: dbz_dx, dbz_dy
    real(8), parameter :: csq = 1d0  ! Speed of light squared

    ! Periodic boundary conditions for Ex
    do i = 1, nx-1
        dbz_dy = (hz(i, 1) - hz(i, ny-1)) / dy
        ex(i, 1) = ex(i, 1) + dt * csq * dbz_dy
        ex(i, ny) = ex(i, 1)
    end do

    ! Periodic boundary conditions for Ey
    do j = 1, ny-1
        dbz_dx = (hz(1, j) - hz(nx-1, j)) / dx
        ey(1, j) = ey(1, j) - dt * csq * dbz_dx
        ey(nx, j) = ey(1, j)
    end do

    ! Update Ex
    do j = 2, ny-1
        do i = 1, nx-1
            dbz_dy = (hz(i, j) - hz(i, j-1)) / dy
            ex(i, j) = ex(i, j) + dt * csq * dbz_dy
        end do
    end do

    ! Update Ey
    do j = 1, ny-1
        do i = 2, nx-1
            dbz_dx = (hz(i, j) - hz(i-1, j)) / dx
            ey(i, j) = ey(i, j) - dt * csq * dbz_dx
        end do
    end do
end subroutine amperemaxwell_fortran
```

---
## **Running the FDTD Simulation**

After compiling the Fortran subroutines, we can call them from Python to run the FDTD simulation.

### **Compile Fortran Code**

In [ ]:
%%bash
f2py --quiet -c faraday_fortran.f90 amperemaxwell_fortran.f -m maxwell --fcompiler=gnu95 --f90flags=-O3

### **Run the Simulation**

In [ ]:
from tqdm.notebook import tqdm
import faraday_fortran
import amperemaxwell_fortran

# Initialize fields with a Gaussian pulse
m, n = 4, 4  # Mode numbers
omega = np.pi  # Angular frequency
ex.fill(0.0)
ey.fill(0.0)
hz = -np.cos(m * np.pi * vy) * np.cos(n * np.pi * vx) * np.cos(omega * (-0.5 * dt))

# Convert arrays to Fortran order
ex = np.asfortranarray(ex)
ey = np.asfortranarray(ey)
hz = np.asfortranarray(hz)

# Time-stepping loop
dt = 0.01  # Time step
for t in tqdm(range(1000)):
    faraday_fortran.faraday_fortran(ex, ey, hz, dx, dy, dt, nx, ny)
    amperemaxwell_fortran.amperemaxwell_fortran(ex, ey, hz, dx, dy, dt, nx, ny)

---
## **Visualization**

To visualize the results, you can use `matplotlib` to plot the fields at different time steps.

**Example: Plot the Magnetic Field (H_z)**

In [ ]:
plt.contourf(vx[:-1], vy[:-1], hz, levels=20, cmap='viridis')
plt.colorbar(label='H_z')
plt.title("Magnetic Field (H_z) at Final Time Step")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

**Output:** A contour plot of the magnetic field $H_z$ at the final time step.

---
## **Key Points**

- **FDTD Method**: A powerful technique for solving Maxwell's equations in the time domain.
- **Staggered Grids**: Electric and magnetic fields are staggered in both space and time.
- **Fortran Optimization**: Using Fortran subroutines can significantly speed up the simulation.
- **Periodic Boundary Conditions**: Applied to simulate an infinite domain.

---